In [22]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import List, Dict, Any, Optional
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
import os
import json
import time
import requests
import asyncio


Agent Build


In [10]:
instructions1="you are a sales agengt working for CompliAI,\
    a compnay that provides a Saas tool for ensuring SOC2 complinace and preparing for audits, powered by AI.\
        You write professional and serious cold emails"

instructions2="you are a humours, enaging sales agent working for compnay complyAi, \
    a company that provides a Saas tool for ensuring SOC2 complinace and preparing for audits, powered by AI.\
        you write witty and engaing cold emails that are likely to get repsonses"

instructions3="You are a busy sales agent working for ComplyAI, a company that provides a Saas tool for ensuring SOC2 complinace and preparing for audits, powered by AI.\
    You write concise and to the point emails"
    

In [11]:
sales_agent1 = Agent(name="Professional",
                    instructions=instructions1,
                    model="gpt-4o-mini")

sales_agent2 = Agent(name="Humorous",
                    instructions=instructions2,
                    model="gpt-4o-mini")

sales_agent3 = Agent(name="Busy Sales Agent",
                    instructions=instructions3,
                    model="gpt-4o-mini")

In [ ]:
# Run the without await
result = Runner.run(
    sales_agent1,  # The agent instance
    "cold email to John Doe at Acme Inc."  # Input prompt
)

# coroutine object
print(result)
#This object is like a paused task waiting to execute

<coroutine object Runner.run at 0x1166427a0>


In [20]:
# Run the agent normally (non-streaming)
result = await Runner.run(
    sales_agent1,  # The agent instance
    "cold email to John Doe at Acme Inc."  # Input prompt
)

# Print the full response at once
print(result.final_output)

Subject: Streamline Your SOC 2 Compliance with AI-Powered Solutions

Hi John,

I hope this message finds you well.

I’m reaching out to introduce you to CompliAI, a leading SaaS solution designed specifically to simplify SOC 2 compliance and streamline the audit preparation process. As compliance requirements continue to evolve, our AI-driven platform provides real-time insights and automation that can significantly reduce manual effort and ensure you're always audit-ready.

Many organizations, like Acme Inc., face challenges in maintaining compliance and managing documentation effectively. With CompliAI, you can:

- Automate repetitive compliance tasks, reducing the time spent on audits.
- Access a centralized repository for all SOC 2-related documentation.
- Gain timely insights and alerts on compliance status, helping you identify and address potential issues proactively.

I would love the opportunity to show you how CompliAI can integrate seamlessly into your existing processes and

/var/folders/3t/c9gcl1zx6j7dgscz2bz8pr100000gp/T/ipykernel_25752/2591778070.py:2: RuntimeWarning: coroutine 'Runner.run' was never awaited
  result = await Runner.run(


In [ ]:
# Run the agent in streaming mode with a given prompt
result = Runner.run_streamed(
    sales_agent1,  # The agent instance responsible for generating the response
    "cold email to John Doe at Acme Inc."  # Input prompt for the agent
)

# Iterate asynchronously over the stream of events produced by the agent
async for event in result.stream_events():
    
    # Check if the event is a raw response event from the model
    if event.type == "raw_response_event":
        
        # Ensure the event contains incremental text output (delta tokens)
        if isinstance(event.data, ResponseTextDeltaEvent):
            
            # Print the text chunk immediately without adding a newline
            # flush=True ensures real-time output in the console
            print(event.data.delta, end="", flush=True)
            

Subject: Streamlining Your SOC 2 Compliance Process

Hi John,

I hope this message finds you well. I’m reaching out to introduce you to CompliAI, a powerful SaaS solution designed to simplify the SOC 2 compliance process and streamline audit preparations through the use of AI-driven insights.

In today’s regulatory environment, achieving and maintaining SOC 2 compliance can be both time-consuming and complex. CompliAI not only helps you stay ahead of evolving compliance requirements but also reduces the burden on your team by automating key tasks, ensuring that you can focus on what really matters—growing your business.

Here are a few ways our solution can add value to Acme Inc.:

- **Automated Documentation:** Generate and manage the necessary documentation quickly and efficiently.
- **Real-Time Monitoring:** Stay informed with ongoing compliance checks that provide immediate alerts for any potential issues.
- **Seamless Audit Preparedness:** Simplify collaboration and data sharing d

In [23]:
message="write a cold sales email"

with trace("parallel Cold Emials"):
    results = await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message)
    )

outputs =[result.final_output for result in results]
for output in outputs:
    print(output+"\n\n")

Subject: Enhance Your SOC 2 Compliance with CompliAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am reaching out from CompliAI, where we specialize in helping organizations like yours streamline their SOC 2 compliance processes.

As you know, maintaining compliance can be a daunting task, often requiring significant resources and meticulous attention to detail. Our AI-powered SaaS tool simplifies this process, providing real-time monitoring, documentation management, and audit preparation to ensure that you stay compliant without the operational overhead.

Here are a few ways CompliAI can add value to your organization:

- **Automated Document Management:** Reduce the manual effort of gathering and organizing documents required for audits.
- **Real-Time Monitoring:** Stay ahead of compliance requirements with constant updates and alerts.
- **Comprehensive Reporting:** Generate detailed reports to facilitate smooth interactions with auditor

In [24]:
sales_agent_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given option\
        imagine you are a customer and pick one that you are most likely to respond to.\
            Don't explain",
    model="gpt-4o-mini"
)

In [26]:
message="write a cold sales email"

with trace("final selection"):
    results = await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message)
    )
    outputs =[result.final_output for result in results]
    email = "cold sales email:\n\n".join(outputs)

    best = await Runner.run(sales_agent_picker,email)
    print(f'best sales email is: \n\n{best.final_output}')



best sales email is: 

Subject: Don't Let SOC 2 Compliance Give You Sleepless Nights 💤

Hi [Recipient's Name],

Are you tired of losing sleep over SOC 2 compliance? 😴 I mean, who wouldn’t want 50 more pillows on their bed instead of worrying about audits? 

At ComplyAi, we've harnessed the powers of AI to make compliance as easy as pie—CEO-approved, low-calorie pie, of course! Our SaaS tool is designed to take the headache out of preparing for audits, leaving you with more time to focus on what really matters: like perfecting your office ping-pong skills. 🏓

Picture this: an audit that practically runs itself, leaving you to wonder why you ever crammed for it like an all-night crammer during finals week. With our solution, your SOC 2 compliance not only becomes a breeze, but you might even enjoy it. (Yes, I said "enjoy" and “compliance” in the same sentence—magic, right?)

Let's set up a time to chat about how ComplyAi can turn your compliance chaos into smooth sailing. Maybe we can ev

Tools - Onwards from here, we will use tool

In [39]:
from sendgrid import content, from_email, to_email
from urllib3 import response


@function_tool
def send_email(body:str):
    """Send out an email with the given body to all sales prospects"""
    sg=sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email=Email(os.environ.get('EMAIL_FROM'))
    to_email=To(os.environ.get('EMAIL_TO'))
    content=Content("text/plain",body)
    mail=Mail(from_email,to_email,"Sales Email",content).get()
    response=sg.client.mail.send.post(request_body=mail)
    return {"status":"success"}


In [40]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193b9260>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [41]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent",tool_description="Write a cold sales emails")
tool1

FunctionTool(name='sales_agent', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193b9440>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [51]:
#agents wrapped as tool
from agents import tool


tool1 = sales_agent1.as_tool(tool_name="sales_agent1",tool_description="Write a cold sales emails")
tool2 = sales_agent2.as_tool(tool_name="sales_agent2",tool_description="Write a cold sales emails")
tool3 = sales_agent3.as_tool(tool_name="sales_agent3",tool_description="Write a cold sales emails")

tools = [tool1,tool2,tool3,send_email]



In [52]:
instructions="you are a sales manger working for complyAI. You use the tools given to you to gnerate cold emails\
    you never generate email yourself, you always use tool\
        you try all 3 sales_agent once before choosing the best one\
            You pik the single best email and uise the send_email to send the best email and only the ebst email to the user"
sales_manger = Agent(name="sales_manager_agent", tools=tools,instructions=instructions, model="gpt-4o-mini")

message = "send a cold sales email addressed to the dear CEO"

with trace("sales manager"):
    result = await Runner.run(sales_manger,message)

In [53]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [54]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(os.environ.get('EMAIL_FROM'))  
    to_email = To(os.environ.get('EMAIL_FROM'))  
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [55]:
tools = [subject_tool, html_tool, send_html_email]

In [56]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1187bcd60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x117e55080>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Function

In [57]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [60]:
tools_subject = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)
#[FunctionTool(name='sales_agent2', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193b8c20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193bade0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent3_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193bb6a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)]
#[Agent(name='Email Manager', handoff_description='Convert an email to HTML and send it', tools=[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193ba200>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193bbf60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='send_html_email', description='Send out an email with the given subject and HTML body to all sales prospects', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_html_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193b9940>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)], mcp_servers=[], mcp_config={}, instructions='You are an email formatter and sender. You receive the body of an email to be sent. You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. Finally, you use the send_html_email tool to send the email with the subject and HTML body.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)]

[FunctionTool(name='sales_agent1', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1193b93a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales emails', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x117e549a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write 

In [62]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools_subject,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Lokenddra"

with trace("Automated SDR1"):
    result = await Runner.run(sales_manager, message)

In [ ]:
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)